<div style='background: linear-gradient(135deg, #0a0a2e, #3d1278, #7F77DD); padding: 40px; border-radius: 14px; text-align: center; color: white;'>
<h1 style='font-size:2.2em; margin:0 0 8px'>PriceDetection: um novíssimo Detector de Preços</h1>
<h2 style='font-weight:400; color:#c8b8f8; margin:0 0 16px'>Coleta de Dados com Python + Faker</h2>
<p style='color:#e0d8ff; max-width:600px; margin:0 auto; line-height:1.7'>
Iremos construir o <strong>coletor de dados</strong>:
um script que simula a chegada de novos preços todo dia e popula o banco automaticamente.
</p>
</div>

##Etapa 2: O projeto

| Etapa | O que acontece |
|-------|----------------|
| 1 | Instalar e conhecer a biblioteca `Faker` |
| 2 | Criar o `Coletor`: classe Python que gera preços |
| 3 | Lógica de negócio: promoções, sazonalidade, variação por mercado |
| 4 | Rodar a coleta e salvar no banco SQLite |
| 5 | Simular 30 dias de histórico de uma vez |
| 6 | Validar os dados com queries SQL e gráficos |
| 7 | Baixar o banco atualizado para o DBeaver |


---
##O início: carregamento do banco de dados.

In [17]:
!pip install faker pandas plotly --quiet

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
import sqlite3
import pandas as pd
import random
import plotly.express as px
import plotly.graph_objects as go
from faker import Faker
from datetime import date, timedelta
from google.colab import files

fake = Faker('pt_BR')
random.seed(42)

In [20]:
# OPÇÃO B — descomente se quiser carregar seu banco da Semana 1:
uploaded = files.upload()   # selecione o priceradar.db
DB_PATH = 'priceradar.db'

Saving priceradar.db to priceradar.db


---
##Parte 1: utilizando o Faker
O Faker gera dados falsos, mas realistas. Vamos usá-lo para simular
variações de preço como se fossem coletas reais.

In [21]:
#uma demonstração de como o Faker consegue gerar informações
print('Nomes de pessoas:')
for _ in range(3):
    print(f'  {fake.name()}')

print('\nDatas aleatórias:')
for _ in range(3):
    print(f'  {fake.date_between(start_date="-30d", end_date="today")}')

print('\nNúmeros decimais com intervalo:')
for _ in range(5):
    print(f'  R$ {fake.pydecimal(left_digits=2, right_digits=2, positive=True, min_value=5, max_value=30)}')

print('\nNo projeto, vou usar o Faker para gerar variações de preço realistas com sazonalidade, promoções e diferenças entre mercados.')

Nomes de pessoas:
  Maria Helena Jesus
  Lucas Gabriel Mendes
  Jade Guerra

Datas aleatórias:
  2026-03-26
  2026-04-20
  2026-04-06

Números decimais com intervalo:
  R$ 26.78
  R$ 24.58
  R$ 24.31
  R$ 12.30
  R$ 20.57

No projeto, vou usar o Faker para gerar variações de preço realistas com sazonalidade, promoções e diferenças entre mercados.


---
## Segunda parte: a classe Coletora
Vou criar uma classe Python chamada `ColetorDePrecos`.
Ela encapsula toda a lógica de geração e inserção de preços no banco.

In [22]:
class ColetorDePrecos:
    """
    Simula a coleta diária de preços em supermercados.

    Regra do jogo:
    - Cada supermercado tem um 'fator de preço' próprio (atacadão é mais barato, comper é mais caro)
    - Preços variam ±3% a cada dia (oscilação normal do mercado)
    - Promoções acontecem com 8% de chance em qualquer produto/mercado
    - Inflação acumulada de ~0.3% ao mês
    """

    # Preço de referência (R$) de cada produto: base para os cálculos
    PRECOS_BASE = {
        1:  22.90,   # Arroz Tio João 5kg
        2:  21.50,   # Arroz Camil 5kg
        3:   8.20,   # Feijão Carioca 1kg
        4:   8.90,   # Feijão Preto 1kg
        5:   6.50,   # Macarrão Espaguete 500g
        6:   6.10,   # Óleo de Soja 900ml
        7:  28.90,   # Azeite Extra Virgem 500ml
        8:   4.50,   # Leite Integral 1L
        9:  18.90,   # Queijo Mussarela 500g
        10:  8.90,   # Manteiga 200g
        11: 15.90,   # Peito de Frango 1kg
        12: 14.50,   # Patinho Moído 500g
        13: 12.90,   # Shampoo Pantene 400ml
        14:  2.50,   # Sabonete Dove 90g
        15:  2.50,   # Detergente Ypê 500ml
        16: 10.90,   # Sabão em Pó OMO 1kg
    }

    # Cada supermercado pratica um nível de preço diferente
    # 1.0 = preço médio de mercado
    # < 1.0 = mais barato ou > 1.0 = mais caro
    FATOR_MERCADO = {
        1: 0.92,   # Atacadão   mais barato (atacado)
        2: 1.05,   # Bretas     preço médio
        3: 1.00,   # Superpão   preço médio
        4: 1.08,   # Comper     um pouco mais caro
        5: 0.95,   # Sam's Club barato (clube de compras)
    }

    def __init__(self, conn: sqlite3.Connection):
        self.conn   = conn
        self.cursor = conn.cursor()

    def _calcular_preco(self,
                        produto_id: int,
                        supermercado_id: int,
                        data: date) -> tuple:
        """
        Calcula o preço de um produto em um mercado numa data.
        Retorna (preco, em_promocao).

        Fórmula:
            preco = base × fator_mercado × inflacao × variacao_diaria × desconto_promo
        """

        preco_base = self.PRECOS_BASE[produto_id]
        fator_mercado = self.FATOR_MERCADO[supermercado_id]

        # Inflação acumulada: +0.3% a cada 30 dias a partir de 2024-01-01
        dias_desde_inicio = (data - date(2024, 1, 1)).days
        fator_inflacao    = 1 + (dias_desde_inicio / 30) * 0.003

        # Variação diária aleatória: ±3%
        variacao = random.uniform(-0.03, 0.03)

        preco = round(
            preco_base * fator_mercado * fator_inflacao * (1 + variacao),
            2
        )

        # Promoção: 8% de chance, desconto entre 10% e 25%
        em_promocao = 0
        if random.random() < 0.08:
            desconto    = random.uniform(0.10, 0.25)
            preco       = round(preco * (1 - desconto), 2)
            em_promocao = 1

        return preco, em_promocao

    def coletar_dia(self, data: date) -> int:
        """
        Coleta preços de todos os produtos em todos os supermercados para uma data específica.
        Retorna o número de registros inseridos.
        """

        # Busca IDs do banco
        produto_ids      = [r[0] for r in self.cursor.execute('SELECT id FROM produtos').fetchall()]
        supermercado_ids = [r[0] for r in self.cursor.execute('SELECT id FROM supermercados').fetchall()]

        registros = []
        for pid in produto_ids:
            for sid in supermercado_ids:
                # Só coleta se o produto tem preço base definido
                if pid in self.PRECOS_BASE:
                    preco, em_promo = self._calcular_preco(pid, sid, data)
                    registros.append((pid, sid, preco, data.isoformat(), em_promo))

        self.cursor.executemany(
            'INSERT INTO precos (produto_id, supermercado_id, preco, data_coleta, em_promocao) '
            'VALUES (?, ?, ?, ?, ?)',
            registros
        )
        self.conn.commit()
        return len(registros)

    def coletar_periodo(self, data_inicio: date, data_fim: date) -> dict:
        """
        Coleta preços para um período inteiro (ex: últimos 30 dias).
        Retorna resumo com total de registros por dia.
        """
        resumo    = {}
        data_atual = data_inicio

        while data_atual <= data_fim:
            n = self.coletar_dia(data_atual)
            resumo[data_atual.isoformat()] = n
            data_atual += timedelta(days=1)

        return resumo


print('Classe ColetorDePrecos definida!')
print()
print('Produtos com preço base cadastrado:', len(ColetorDePrecos.PRECOS_BASE))
print('Fatores de mercado:')
for mercado_id, fator in ColetorDePrecos.FATOR_MERCADO.items():
    nivel = 'barato' if fator < 1.0 else ('médio' if fator == 1.0 else 'caro')
    print(f'  Mercado {mercado_id}: fator {fator} → {nivel}')

Classe ColetorDePrecos definida!

Produtos com preço base cadastrado: 16
Fatores de mercado:
  Mercado 1: fator 0.92 → barato
  Mercado 2: fator 1.05 → caro
  Mercado 3: fator 1.0 → médio
  Mercado 4: fator 1.08 → caro
  Mercado 5: fator 0.95 → barato


### Qual a lógica das nossas definições de preço?

```
preco_final = preco_base
              × fator_mercado     (Atacadão é 8% mais barato que a média)
              × fator_inflacao    (+0.3% a cada 30 dias)
              × variacao_diaria   (±3% aleatório por dia)
              × desconto_promo    (−10% a −25%, com 8% de chance)
```

Isso simula como os preços funcionam na vida real:
cada mercado tem seu posicionamento, os preços flutuam diariamente,
a inflação corrói o poder de compra ao longo do tempo,
e promoções aparecem de forma imprevisível.

---
## Terceira parte: rodando a coleta

In [23]:
conn = sqlite3.connect(DB_PATH)
coletor = ColetorDePrecos(conn)

hoje = date.today()
n = coletor.coletar_dia(hoje)

print(f'Coleta do dia {hoje} concluída!')
print(f'{n} registros inseridos ({len(ColetorDePrecos.PRECOS_BASE)} produtos × 5 supermercados)')

#Conferir no banco
df_hoje = pd.read_sql(
    '''
    SELECT p.nome AS produto, s.nome AS supermercado,
           pr.preco, pr.em_promocao
    FROM precos pr
    JOIN produtos p ON p.id = pr.produto_id
    JOIN supermercados s ON s.id = pr.supermercado_id
    WHERE pr.data_coleta = DATE('now')
    ORDER BY p.nome, pr.preco
    ''', conn
)
print(f'\nPrévia dos preços de hoje ({len(df_hoje)} linhas):')
display(df_hoje.head(20))

Coleta do dia 2026-04-25 concluída!
80 registros inseridos (16 produtos × 5 supermercados)

Prévia dos preços de hoje (160 linhas):


,produto,supermercado,preco,em_promocao
0,Arroz Branco,Sam's Club,17.58,1
1,Arroz Branco,Atacadão,18.67,1
2,Arroz Branco,Atacadão,19.79,1
3,Arroz Branco,Atacadão,21.05,0
4,Arroz Branco,Sam's Club,21.96,0
5,Arroz Branco,Sam's Club,23.01,0
6,Arroz Branco,Atacadão,23.37,0
7,Arroz Branco,Superpão,23.41,0
8,Arroz Branco,Superpão,23.59,0
9,Arroz Branco,Sam's Club,23.99,0


In [24]:
print('Simulando 30 dias de histórico de preços...\n')

data_inicio = hoje - timedelta(days=29)
data_fim = hoje - timedelta(days=1)   # hoje já foi coletado acima

resumo = coletor.coletar_periodo(data_inicio, data_fim)

total_registros = sum(resumo.values())
print(f'Simulação concluída!')
print(f'   Período: {data_inicio} → {hoje}')
print(f'   Dias simulados: {len(resumo) + 1}')
print(f'   Total de registros: {total_registros + n:,}')

# Verificar no banco
total_banco = pd.read_sql('SELECT COUNT(*) AS total FROM precos', conn).iloc[0,0]
print(f'   Registros no banco: {total_banco:,}')

Simulando 30 dias de histórico de preços...

Simulação concluída!
   Período: 2026-03-27 → 2026-04-25
   Dias simulados: 30
   Total de registros: 2,400
   Registros no banco: 4,800


---
##Quarta parte: validando os dados

In [25]:
df_dias = pd.read_sql(
    'SELECT data_coleta, COUNT(*) AS registros FROM precos GROUP BY data_coleta ORDER BY data_coleta',
    conn
)

fig = px.bar(
    df_dias, x='data_coleta', y='registros',
    title='Registros coletados por dia',
    labels={'data_coleta': 'Data', 'registros': 'Nº de registros'},
    color_discrete_sequence=['#7F77DD']
)
fig.update_layout(plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
                  font_color='white', title_font_size=15)
fig.show()
print(f'Cada dia tem {df_dias["registros"].mean():.0f} registros em média')

Cada dia tem 160 registros em média


In [26]:
# ── Histórico de preços: Arroz Tio João em todos os mercados ───────────────
df_hist = pd.read_sql(
    '''
    SELECT pr.data_coleta, s.nome AS supermercado, pr.preco, pr.em_promocao
    FROM precos pr
    JOIN produtos p ON p.id = pr.produto_id
    JOIN supermercados s ON s.id = pr.supermercado_id
    WHERE p.nome = 'Arroz Branco' AND p.marca = 'Tio João'
    ORDER BY pr.data_coleta
    ''', conn
)

fig = px.line(
    df_hist, x='data_coleta', y='preco', color='supermercado',
    title='Histórico de preços: Arroz Branco Tio João 5kg',
    labels={'data_coleta': 'Data', 'preco': 'Preço (R$)', 'supermercado': 'Supermercado'},
    markers=True
)
fig.update_layout(plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
                  font_color='white', title_font_size=14)
fig.show()

In [27]:
#Quantas promoções foram detectadas?
df_promo = pd.read_sql(
    '''
    SELECT
        s.nome AS supermercado,
        COUNT(*) AS total_promocoes,
        ROUND(AVG(pr.preco), 2) AS preco_medio_promo
    FROM precos pr
    JOIN supermercados s ON s.id = pr.supermercado_id
    WHERE pr.em_promocao = 1
    GROUP BY s.id
    ORDER BY total_promocoes DESC
    ''', conn
)

print('Promoções por supermercado (últimos 30 dias):')
display(df_promo)

total_promo  = df_promo['total_promocoes'].sum()
total_coleta = pd.read_sql('SELECT COUNT(*) AS n FROM precos', conn).iloc[0,0]
print(f'\n{total_promo} promoções em {total_coleta} registros = {total_promo/total_coleta*100:.1f}% do total')
print('(Esperado: ~8% pois a probabilidade configurada foi 8%)')

Promoções por supermercado (últimos 30 dias):


,supermercado,total_promocoes,preco_medio_promo
0,Atacadão,78,10.32
1,Bretas,74,11.57
2,Superpão,69,10.69
3,Sam's Club,67,8.98
4,Comper,58,11.53



346 promoções em 4800 registros = 7.2% do total
(Esperado: ~8% pois a probabilidade configurada foi 8%)


In [28]:
#Ranking de mercados por preço médio histórico
df_rank = pd.read_sql(
    '''
    SELECT
        s.nome                          AS supermercado,
        ROUND(AVG(pr.preco), 2)         AS preco_medio,
        COUNT(DISTINCT pr.produto_id)   AS produtos,
        COUNT(*)                        AS coletas
    FROM precos pr
    JOIN supermercados s ON s.id = pr.supermercado_id
    WHERE pr.em_promocao = 0    -- excluindo promoções para comparar fairly
    GROUP BY s.id
    ORDER BY preco_medio ASC
    ''', conn
)

fig = px.bar(
    df_rank, x='supermercado', y='preco_medio',
    title='Ranking de supermercados por preço médio (sem promoções)',
    text='preco_medio',
    color='preco_medio',
    color_continuous_scale='Viridis_r'
)
fig.update_traces(texttemplate='R$ %{text:.2f}', textposition='outside')
fig.update_layout(plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
                  font_color='white', showlegend=False, title_font_size=14)
fig.show()

In [29]:
#Inflação acumulada: preço médio ao longo do tempo
df_inflacao = pd.read_sql(
    '''
    SELECT
        pr.data_coleta,
        ROUND(AVG(pr.preco), 4) AS preco_medio_dia
    FROM precos pr
    WHERE pr.em_promocao = 0
    GROUP BY pr.data_coleta
    ORDER BY pr.data_coleta
    ''', conn
)

# Calcula variação percentual acumulada em relação ao primeiro dia
preco_base_ref = df_inflacao['preco_medio_dia'].iloc[0]
df_inflacao['variacao_pct'] = (
    (df_inflacao['preco_medio_dia'] - preco_base_ref) / preco_base_ref * 100
).round(2)

fig = px.area(
    df_inflacao, x='data_coleta', y='variacao_pct',
    title='Variação do preço médio ao longo do período (%)',
    labels={'data_coleta': 'Data', 'variacao_pct': 'Variação (%)'},
    color_discrete_sequence=['#7F77DD']
)
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.5)
fig.update_layout(plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
                  font_color='white', title_font_size=14)
fig.show()
print(f'Variação acumulada no período: {df_inflacao["variacao_pct"].iloc[-1]:+.2f}%')

Variação acumulada no período: +1.92%


In [30]:
#Boxplot: dispersão de preços por supermercado
df_box = pd.read_sql(
    '''
    SELECT s.nome AS supermercado, pr.preco
    FROM precos pr
    JOIN supermercados s ON s.id = pr.supermercado_id
    WHERE pr.em_promocao = 0
    ''', conn
)

fig = px.box(
    df_box, x='supermercado', y='preco',
    title='Dispersão de preços por supermercado (todos os produtos)',
    color='supermercado'
)
fig.update_layout(plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
                  font_color='white', showlegend=False, title_font_size=14)
fig.show()

---
## Quinta parte: Simular um novo dia (reutilização)

In [31]:
#Como rodar toda manhã para atualizar o banco
print('Simulando coleta de amanhã ...')
amanha = hoje + timedelta(days=1)
n = coletor.coletar_dia(amanha)

print(f'{n} novos registros inseridos para {amanha}')

#Mostra os produtos em promoção amanhã
df_promos_amanha = pd.read_sql(
    f'''
    SELECT p.nome AS produto, s.nome AS supermercado,
           pr.preco,
           ROUND((pr.preco / MIN(pr2.preco) - 1) * 100, 1) AS desconto_estimado_pct
    FROM precos pr
    JOIN produtos p ON p.id  = pr.produto_id
    JOIN supermercados s ON s.id  = pr.supermercado_id
    LEFT JOIN precos pr2
           ON pr2.produto_id = pr.produto_id
          AND pr2.data_coleta = DATE('{amanha.isoformat()}', '-1 day')
    WHERE pr.data_coleta = '{amanha.isoformat()}'
      AND pr.em_promocao = 1
    GROUP BY pr.id
    ''', conn
)

if len(df_promos_amanha) > 0:
    print(f'\n{len(df_promos_amanha)} promoções em {amanha}:')
    display(df_promos_amanha)
else:
    print(f'\nNenhuma promoção detectada para {amanha}')

Simulando coleta de amanhã ...
80 novos registros inseridos para 2026-04-26

6 promoções em 2026-04-26:


,produto,supermercado,preco,desconto_estimado_pct
0,Feijão Preto,Sam's Club,7.18,-16.7
1,Óleo de Soja,Bretas,5.48,0.4
2,Óleo de Soja,Sam's Club,5.53,1.3
3,Manteiga,Comper,8.64,-1.3
4,Peito de Frango,Sam's Club,13.51,-12.8
5,Detergente,Comper,2.33,19.5


---
##Sexta parte: baixar o banco atualizado

In [32]:
print('Resumo do banco atuazalido:\n')

for tabela in ['categorias', 'supermercados', 'produtos', 'precos']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {tabela}', conn).iloc[0,0]
    print(f'  {tabela:<20}: {n:>6,} registros')

periodo = pd.read_sql(
    'SELECT MIN(data_coleta) AS inicio, MAX(data_coleta) AS fim FROM precos', conn
).iloc[0]
print(f'\n  Período coberto: {periodo["inicio"]} → {periodo["fim"]}')

conn.commit()
conn.close()
print('\nBanco fechado e pronto para download!')

# Baixar
files.download('priceradar.db')
print('Download iniciado!')

Resumo do banco atuazalido:

  categorias          :      6 registros
  supermercados       :      5 registros
  produtos            :     16 registros
  precos              :  4,880 registros

  Período coberto: 2026-03-27 → 2026-04-26

Banco fechado e pronto para download!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download iniciado!
